# LLM Toy Example

The following code is generated by LLM. This code act as a reference check. If you understand all the code in this notebook then you understand the make of LLM.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [2]:
# ==========================================
# 1. Training text
# ==========================================
text = """
mary had a little lamb
its fleece was white as snow
and everywhere that mary went
the lamb was sure to go
mary had a little lamb
the lamb was sure to go
"""

tokens = text.lower().split()
vocab = sorted(set(tokens))

stoi = {word: i for i, word in enumerate(vocab)}
itos = {i: word for word, i in stoi.items()}

data = torch.tensor([stoi[w] for w in tokens])

vocab_size = len(vocab)


In [3]:
data

tensor([10,  6,  0,  9,  8,  7,  4, 16, 18,  2, 11,  1,  3, 13, 10, 17, 14,  8,
        16, 12, 15,  5, 10,  6,  0,  9,  8, 14,  8, 16, 12, 15,  5])

In [4]:
# ==========================================
# 2. Hyperparameters
# ==========================================
batch_size = 16
block_size = 8
d_model = 32
n_layers = 2
learning_rate = 0.003
max_iters = 1000

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 3. Create training batches
# ==========================================
def get_batch():
    starts = torch.randint(
        0, len(data) - block_size, (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size] for i in starts
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1] for i in starts
    ])

    return x.to(device), y.to(device)


# ==========================================
# 4. SINGLE-HEAD causal self-attention
# ==========================================
class SingleHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()

        # Three learnable transformations
        self.query = nn.Linear(d_model, d_model, bias=False)
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)

        # Lower triangular mask: block future tokens
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.output = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape

        # Step 1: Create Q, K, V
        Q = self.query(x)  # (B, T, C)
        K = self.key(x)    # (B, T, C)
        V = self.value(x)  # (B, T, C)

        # Step 2: Calculate attention scores
        scores = Q @ K.transpose(-2, -1)

        # Scale scores for numerical stability
        scores = scores / (C ** 0.5)

        # Step 3: Prevent looking at future tokens
        scores = scores.masked_fill(
            self.mask[:T, :T] == 0,
            float("-inf")
        )

        # Step 4: Scores -> attention probabilities
        weights = F.softmax(scores, dim=-1)

        # Step 5: Weighted combination of values
        out = weights @ V

        return self.output(out)


# ==========================================
# 5. Feed-forward network
# ==========================================
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        return self.net(x)


# ==========================================
# 6. Transformer decoder block
# ==========================================
class DecoderBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attention = SingleHeadAttention()

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward()

    def forward(self, x):
        # Attention + residual connection
        x = x + self.attention(self.ln1(x))

        # Feed-forward + residual connection
        x = x + self.ffn(self.ln2(x))

        return x


# ==========================================
# 7. Decoder-only language model
# ==========================================
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size, d_model
        )

        self.position_embedding = nn.Embedding(
            block_size, d_model
        )

        self.blocks = nn.Sequential(
            *[DecoderBlock() for _ in range(n_layers)]
        )

        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Token identity + position
        tok_emb = self.token_embedding(idx)
        pos = torch.arange(T, device=idx.device)
        pos_emb = self.position_embedding(pos)

        x = tok_emb + pos_emb

        # Transformer decoder blocks
        x = self.blocks(x)
        x = self.final_ln(x)

        # Predict vocabulary scores at each position
        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, vocab_size),
                targets.reshape(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=20):
        self.eval()

        for _ in range(max_new_tokens):
            context = idx[:, -block_size:]

            logits, _ = self(context)

            # Use the last position to predict the next token
            logits = logits[:, -1, :]

            probabilities = F.softmax(logits, dim=-1)

            next_token = torch.multinomial(
                probabilities, num_samples=1
            )

            idx = torch.cat([idx, next_token], dim=1)

        return idx


# ==========================================
# 8. Train
# ==========================================
model = MiniGPT().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate
)

print("Vocabulary size:", vocab_size)
print("Parameters:", sum(p.numel() for p in model.parameters()))

for step in range(max_iters + 1):
    model.train()

    x, y = get_batch()
    _, loss = model(x, y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        print(f"Step {step:4d} | Loss: {loss.item():.4f}")



Vocabulary size: 19
Parameters: 26771
Step    0 | Loss: 3.2381
Step  200 | Loss: 0.0689
Step  400 | Loss: 0.0268
Step  600 | Loss: 0.0425
Step  800 | Loss: 0.0564
Step 1000 | Loss: 0.0379


In [5]:
# ==========================================
# 9. Generate text
# ==========================================
start = torch.tensor(
    [[stoi["mary"]]], dtype=torch.long, device=device
)

generated = model.generate(start, max_new_tokens=10)

result = [itos[i] for i in generated[0].tolist()]

print("\nGenerated poem:")
print(" ".join(result))


Generated poem:
mary went the lamb was sure to go mary had a


In [6]:
# ==========================================
# 9. Generate text
# ==========================================
prompt = "mary"
# Lowercase, split into words, and map each to its integer ID
prompt_tokens = [stoi[w] for w in prompt.lower().split()]

start = torch.tensor(
    [prompt_tokens], dtype=torch.long, device=device
)

generated = model.generate(start, max_new_tokens=8)

result = [itos[i] for i in generated[0].tolist()]

print("\nGenerated text:")
print(" ".join(result))


Generated text:
mary had a little lamb its fleece was white
